# Machine Learning Classification of Player Feedback from Steam Reviews

**Student:** Robert Mayfield  
**Project:** Udacity AI Masters Capstone - Applied Machine Learning  
**Series:** AI Game Director Studio (Project 3 of 7)

## 1. Project Overview

This project trains a supervised machine learning classifier to predict whether a player review reflects a positive or negative experience. The classifier takes raw review text as input and produces a binary sentiment signal, converting unstructured player language into a structured output that downstream systems can act on.

### Training data

The model is trained on Steam game reviews using the `voted_up` label, a direct binary recommendation provided by the reviewer. Steam reviews represent the largest publicly available labeled dataset of player generated feedback in natural language, covering 15,437,471 reviews across 8,183 games (forgemaster, 2024).

### What this project produces

The trained classifier and TF-IDF vectorizer are saved as reusable artifacts. Any system that receives player feedback text can load these artifacts and produce a sentiment signal without retraining.

### References

forgemaster. (2024). *Steam Reviews Dataset* [Data set]. Kaggle. https://www.kaggle.com/datasets/forgemaster/steam-reviews-dataset

## 2. Dataset Loading and Inspection

In [2]:
import os
import json
import pickle
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.inspection import inspect_dataframe

RANDOM_SEED  = 42
TARGET_SAMPLE = 100_000
RAW_ZIP   = Path('../data/raw/archive.zip')
PROCESSED = Path('../data/processed/steam_reviews_sample.csv')
SAMPLE    = Path('../data/sample/steam_reviews_1k.csv')

print(f"Raw archive present: {RAW_ZIP.exists()}")
print(f"Processed sample present: {PROCESSED.exists()}")

Raw archive present: True
Processed sample present: True


In [3]:
if PROCESSED.exists():
    print("Processed sample found - skipping extraction.")
    df = pd.read_csv(PROCESSED)
else:
    print("Building processed sample from archive...")
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)

    frames = []
    with zipfile.ZipFile(RAW_ZIP) as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        print(f"CSV files in archive: {len(csv_files)}")
        for name in csv_files:
            with z.open(name) as fh:
                chunk = pd.read_csv(fh, usecols=['appid', 'review', 'voted_up'],
                                    low_memory=False)
                frames.append(chunk)
                print(f"  Loaded {name}: {len(chunk):,} rows")

    raw = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows: {len(raw):,}")

    raw = raw.dropna(subset=['review', 'voted_up']).reset_index(drop=True)
    raw['voted_up'] = raw['voted_up'].astype(bool)

    pos = raw[raw['voted_up'] == True]
    neg = raw[raw['voted_up'] == False]
    n_pos = int(TARGET_SAMPLE * len(pos) / len(raw))
    n_neg = int(TARGET_SAMPLE * len(neg) / len(raw))
    sample = pd.concat([
        pos.sample(n=min(len(pos), n_pos), random_state=RANDOM_SEED),
        neg.sample(n=min(len(neg), n_neg), random_state=RANDOM_SEED),
    ]).reset_index(drop=True)

    sample.to_csv(PROCESSED, index=False)
    print(f"Saved: {len(sample):,} rows to {PROCESSED}")
    df = sample

print(f"\nDataset loaded: {len(df):,} rows, {df.shape[1]} columns")

Processed sample found - skipping extraction.

Dataset loaded: 99,999 rows, 3 columns


In [4]:
if not SAMPLE.exists():
    SAMPLE.parent.mkdir(parents=True, exist_ok=True)
    df.sample(n=1000, random_state=RANDOM_SEED).to_csv(SAMPLE, index=False)
    print(f"Saved 1k sample → {SAMPLE}")
else:
    print("1k sample already exists.")

1k sample already exists.


In [5]:
inspect_dataframe(df)

SHAPE AND STRUCTURE
------------------------------------------------------------
Rows:    99,999
Columns: 3

COLUMN NAMES AND DATA TYPES
------------------------------------------------------------
appid       int64
voted_up     bool
review        str

MEMORY USAGE
------------------------------------------------------------
Total: 32.03 MB

MISSING VALUES
------------------------------------------------------------
No missing values found.

DUPLICATE ROWS
------------------------------------------------------------
Duplicate rows: 2,882

DESCRIPTIVE STATISTICS (NUMERIC)
------------------------------------------------------------
              appid
count  9.999900e+04
mean   3.050983e+05
std    2.141970e+05
min    1.000000e+01
25%    2.037700e+05
50%    2.836800e+05
75%    4.183700e+05
max    1.046030e+06

DESCRIPTIVE STATISTICS (CATEGORICAL)
------------------------------------------------------------
       review
count   99999
unique  89199
top      good
freq      592

UNIQUE VALU

In [6]:
counts = df['voted_up'].value_counts()
pct    = df['voted_up'].value_counts(normalize=True) * 100
print("Class distribution")
print(f"  Positive (voted_up=True):  {counts[True]:>7,}  ({pct[True]:.1f}%)")
print(f"  Negative (voted_up=False): {counts[False]:>7,}  ({pct[False]:.1f}%)")

Class distribution
  Positive (voted_up=True):   86,944  (86.9%)
  Negative (voted_up=False):  13,055  (13.1%)


The sample contains 99,999 rows across 3 columns (appid, review, voted_up) with no missing values, confirming the dataset is clean on ingestion. Memory usage is 32.03 MB, which is manageable in memory without chunking.

The dataset is strongly class imbalanced: 86,944 positive reviews (86.9%) versus 13,055 negative reviews (13.1%), a ratio of roughly 6.7 to 1. This imbalance reflects real Steam user behavior but means a naive classifier that predicts positive for every review would achieve 86.9% accuracy without learning anything meaningful. Evaluation will therefore emphasize macro F1 score, which weights each class equally regardless of support.

There are 2,882 duplicate rows in the sample. These are likely review texts that appear across multiple games rather than data entry errors. They will be dropped in the data quality review to avoid inflating performance on repeated text.

## 3. Data Quality Review

## 4. Text and Feature Preprocessing

## 5. Model Selection

## 6. Train/Test Split

## 7. Baseline Model

## 8. Model Evaluation

## 9. Error Analysis

## 10. Model Comparison and Selection

## 11. Limitations, Bias, and Responsible Use

## 12. Future Integration with AI Game Director Studio

## 13. Final Notebook Summary